# PitchTrack — f0, impulse pattern, melody and tonal system

The pitch track has several stages. This notebook walks through them on a single
recording:

1. **f0** — the fundamental frequency per frame (`PitchTrack`).
2. **Impulse pattern** — one impulse per pitch period (`ImpulsePattern`).
3. **Melody / notes** — segment the f0 into notes (`PitchTrack.notes`).
4. **Tonal system** — the best-matching scale (`PitchTrack.tonal_system`).

Everything is shown together in the interactive `pitch_player`.

> Requires `bader-comsar` plus `soundfile`.

In [ ]:
from comsar import PitchTrack, ImpulsePattern, pitch_player

## Analysis windows

Windowing is given in time units (milliseconds + overlap fraction), so any
sample rate works. 25 ms windows with 60 % overlap correspond to the historical
1102-sample window with a 441-sample hop at 44.1 kHz.

In [ ]:
tt = PitchTrack(window_ms=25.0, overlap=0.6)

## Extract f0 from a single recording

`extract` returns a `TrackResult`; `.features` is a DataFrame indexed by
`time_s` with the columns `Pitch` (f0 in Hz) and `SPL`. Unvoiced / silent
frames have `Pitch = 0`.

In [ ]:
WAV = "CAM-5_SARA PANH.wav"
result = tt.extract(WAV)
result.features.head()

## Save f0 as CSV

Written next to the audio file. `EXCEL_STYLE` uses semicolons + decimal commas
for European Excel (set to `False` for a standard comma CSV; read the Excel
style back with `pd.read_csv(path, sep=";", decimal=",", index_col=0)`).

In [ ]:
EXCEL_STYLE = True
sep, dec = (";", ",") if EXCEL_STYLE else (",", ".")

CSV_FILE = WAV.rsplit(".", 1)[0] + "_f0.csv"
result.features.to_csv(CSV_FILE, sep=sep, decimal=dec)
print(f"saved {len(result.features)} frames to {CSV_FILE}")

## ImpulsePattern

The **impulse pattern** turns the waveform into a sequence of impulses, one per
pitch period `T = 1/f0`. Each impulse onset is the **rising zero crossing before
the strongest amplitude** within `T`; only parts above `dbmin` (SPL, default
52 dB) are analysed, and each impulse stores the peak `amplitude` until the next
one. The result is a list — like the pitch track — of `[time_s, amplitude]`.

In [ ]:
ires = ImpulsePattern(dbmin=52.0).extract(WAV, result)
ires.impulses.head()

In [ ]:
# Save the impulse list (time + amplitude) as CSV next to the audio file.
IMP_CSV = WAV.rsplit(".", 1)[0] + "_impulses.csv"
ires.impulses.to_csv(IMP_CSV, sep=sep, decimal=dec, index=False)
print(f"saved {len(ires.impulses)} impulses to {IMP_CSV}")

## Melody / notes

`PitchTrack.notes` segments the f0 track into notes (runs of stable pitch). It
returns a DataFrame — like the pitch track — with `start_s`, `stop_s`,
`duration_s`, `frequency` (Hz), `cent` (above `f0`) and `midi`.

In [ ]:
notes = tt.notes(result, minlen=15, mindev=60)
print(f"{len(notes)} notes")
notes.head(8)

In [ ]:
# Save the melody (notes) as CSV next to the audio file.
NOTES_CSV = WAV.rsplit(".", 1)[0] + "_notes.csv"
notes.to_csv(NOTES_CSV, sep=sep, decimal=dec, index=False)
print("saved", NOTES_CSV)

## Tonal system

`PitchTrack.tonal_system` accumulates the measured pitches into one octave and
correlates them with 900+ theoretical scales (`scales.csv`). It returns the
best-matching scales, the measured one-octave distribution and the notes.

In [ ]:
ts = tt.tonal_system(result, minlen=15, mindev=60)
print("fundamental: %.1f Hz" % ts.fundamental)
print("best scale:", ts.best["name"], "(correlation %.3f)" % ts.best["correlation"])
ts.scales[["rank", "name", "correlation"]].head()

## Pitch player — all stages together

The player shows the grey waveform, the raw f0 track (blue) on a logarithmic
frequency axis, the **impulses** as orange vertical lines, the **melody notes**
as green horizontal bars, and the **tonal system** as teal reference lines
(scale degrees repeated over every octave). Press **Play** to listen; the cursor
follows the audio. Use the **mouse wheel** / Zoom buttons to zoom horizontally,
**drag** to pan and **click** to seek. Every layer is toggleable via the
legend.

In [ ]:
pitch_player(WAV, result, impulses=ires, notes=notes, tonal_system=ts)